# Corn store-or-sell: the whole process, step by step

**The question.** On **October 15**, should a Central Illinois corn holder **store the corn until April 15**,
hedged by selling May corn futures, or **sell now and put the cash in 6-month T-bills**? With the hedge in
place, storing pays only if the gap between May futures and the local cash price narrows by more than the
storage cost plus the T-bill interest given up. This notebook follows every step, from the raw files to an
out-of-sample test.

The cash-bid data covers **six states** (IL, IA, NE, SD, MN, IN); the decision analysis (Parts 3-5) uses the two Central
Illinois regions for now.

**How to read it.** Every step has three main code cells: **a) raw table → b) cleaning code → c) clean table**.
The code is the real logic from the numbered scripts in `notebooks/` (shortened where noted), re-run here so you
can see each table change. All files are loaded once, in Setup. Each result is then checked against the file the script saved.

*Read-only: every cell only **reads** files. Nothing in `data/`, the scripts, `docs/` or `outputs/` is changed.
Plan: [README](../README.md). Every decision: [processing log](../docs/processing_log.md). Column meanings:
[data dictionary](../docs/data_dictionary.md).*

## Pipeline map

| Source | Raw file (`data/raw/`) | Script | Clean file | Used in |
|---|---|---|---|---|
| USDA MyMarketNews grain-bid reports for 6 states: IL 3192, IA 2850, NE 3225, SD 3186, MN 3049, IN 3463 (API, one month per pull) | IL: `3192_YYYY-MM.csv`; others: `usda_bids/<STATE>/<report_id>_YYYY-MM.csv`; all listed in `usda_bids/manifest.csv` (71 files) | `03_build_silver.py` | `data/clean/corn_bids_current.csv` | `06` → table `corn_bids` → `07` (Oct and Apr cash price, Illinois) |
| CBOT May corn settlements, typed by hand from Barchart | `may_futures.csv` | `04_clean_futures.py` | `data/clean/may_futures.csv` | `06` → table `may_futures` → `07` (hedge prices) |
| FRED series DTB6, 6-month T-bill (download) | `fred_dtb6.csv` | `05_tbill_rates.py` | `data/clean/tbill_decision.csv` | `06` → table `tbill_decision` → `07` (interest) |
| farmdoc daily (commercial) and Iowa Farm Bureau (on-farm), typed by hand | `storage_costs.csv` | none | none (read directly) | `07` (storage cases) |
| the 3 clean CSVs | | `06_build_db.py` | `data/clean/corn.db` (SQLite) | `07` |
| `corn.db` + `storage_costs.csv` | | `07_build_gold.py` | `data/gold/gold_yearly.csv` + table `gold_yearly` | `08`, `09` |
| `gold_yearly.csv` | | `08_charts.py` | `outputs/scatter_gap.png`, `outputs/bars_by_year.png` | Part 4 |
| `gold_yearly.csv` | | `09_walk_forward.py` | `outputs/walk_forward_predictions.csv`, `outputs/scoreboard.csv` | Part 5 |

**Run order:** `03` → `04` → `05` → `06` → `07` → `08` → `09`.
`06` rebuilds `corn.db` from scratch (with `--overwrite`), which also removes the gold table, so `07` must always run after it.

## Setup

**All file loading happens here, once**, so the steps below only show tables and logic. Every file is only **read**;
the database is opened **read-only** (`mode=ro`).

- `show(df)`: shape, first 5 rows, and missing values per column (a blank text cell counts as missing).
- `explore(df, cols)`: for each column, its type, number of distinct values, missing count and most common values.
- `summary(raw, clean, same)`: the one-line "rows X → Y, columns X → Y" after each step, plus whether the notebook's
  code reproduced the saved file exactly.

In [1]:
from pathlib import Path
import hashlib
import re
import sqlite3
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()                                              # find the project folder from anywhere
while not (ROOT / "notebooks/09_walk_forward.py").exists():
    ROOT = ROOT.parent
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
CASES = ("low", "mid", "high")                                 # the three storage-cost cases

def n_missing(df):                                             # NaN or blank text, per column
    return (df.isna() | df.astype(str).apply(lambda s: s.str.strip().eq(""))).sum()

def show(df, n=5):                                             # shape + first n rows + missing values
    miss = n_missing(df)
    print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
    display(df.head(n))
    print("missing per column:", "none" if miss.sum() == 0 else miss[miss > 0].to_dict())

def summary(raw, clean, same=None):                            # the one-line before -> after
    check = "" if same is None else f"   (code above reproduces the saved file exactly: {same})"
    print(f"rows {len(raw):,} → {len(clean):,}, columns {raw.shape[1]} → {clean.shape[1]}{check}")

def explore(df, cols):                                         # one row per column: what is in it?
    return pd.DataFrame({"type": df[cols].dtypes.astype(str),
                         "distinct values": df[cols].nunique(),
                         "missing": n_missing(df[cols]),
                         "most common (rows)": [", ".join(f"{k} ({v:,})" for k, v in df[c].value_counts().head(3).items())
                                                for c in cols]})

def text(rel):                                                 # a file's exact text, for the byte checks
    return (ROOT / rel).read_text()

# Raw files (data/raw/ is evidence: read only).
# Cash bids: every file in the manifest (6 states), stacked with a state column, as in 03 lines 45-54
STATES = ["IL", "IA", "NE", "SD", "MN", "IN"]
manifest = pd.read_csv(ROOT / "data/raw/usda_bids/manifest.csv")              # file, rows, sha256 per state-month
files = manifest[manifest["file"].notna()]                                    # a month with no data has no file
raw_bids = pd.concat([pd.read_csv(ROOT / f).assign(state=st) for f, st in zip(files["file"], files["state"])],
                     ignore_index=True)
raw_futures = pd.read_csv(ROOT / "data/raw/may_futures.csv", dtype=str, keep_default_na=False)   # as text, like 04
raw_tbill = pd.read_csv(ROOT / "data/raw/fred_dtb6.csv", dtype=str, keep_default_na=False)       # as text, like 05
raw_storage = pd.read_csv(ROOT / "data/raw/storage_costs.csv")

# Clean files saved by the scripts (MISSING = blank)
clean_bids = pd.read_csv(ROOT / "data/clean/corn_bids_current.csv", na_values="MISSING")
clean_futures = pd.read_csv(ROOT / "data/clean/may_futures.csv")
clean_tbill = pd.read_csv(ROOT / "data/clean/tbill_decision.csv")
gold = pd.read_csv(ROOT / "data/gold/gold_yearly.csv")
scoreboard = pd.read_csv(ROOT / "outputs/scoreboard.csv")

# Database (read-only) and the list of its tables with keys, read from SQLite itself
con = sqlite3.connect(f"file:{ROOT / 'data/clean/corn.db'}?mode=ro", uri=True)
db_tables = pd.DataFrame([{
    "table": t,
    "rows": con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0],
    "primary key": ", ".join(c[1] for c in sorted(con.execute(f"PRAGMA table_info({t})"), key=lambda c: c[5]) if c[5]),
    "foreign keys": "; ".join(f"{f[3]} → {f[2]}({f[4]})" for f in con.execute(f"PRAGMA foreign_key_list({t})")) or "none",
} for t in ["calendar", "corn_bids", "may_futures", "tbill_decision", "gold_yearly"]])

# Part 1: Clean each source

Each source has three main code cells: **a) raw table → b) cleaning code → c) clean table**, ending with
"rows X → Y, columns X → Y". The cash bids (6 states), the biggest and messiest source, also get two **data exploration** cells (a2, c2).

## 1.1 Cash bids, 6 states (`03_build_silver.py`)

**a) Raw table.** USDA MyMarketNews publishes a daily grain-bid report for each state: every bid for corn, soybeans and
wheat, by region and buyer type, for delivery now and for later. Six states are pulled, October and April months only:

| State | USDA report | Report name |
|---|---|---|
| IL | 3192 | Illinois Grain Bids |
| IA | 2850 | Iowa Daily Cash Grain Bids |
| NE | 3225 | Nebraska Daily Elevator Grain Bids |
| SD | 3186 | South Dakota Daily Grain Bids |
| MN | 3049 | Southern Minnesota Daily Grain Bids |
| IN | 3463 | Indiana Grain Bids |

Range: **Oct 2020 → Apr 2026** (decision years 2020-2025); the API has no bid data before mid-2020. Every file (one per
state-month, saved exactly as the API returned it, 55 columns) is listed in `data/raw/usda_bids/manifest.csv` with its row
count and SHA-256 checksum. `03` reads the files from the manifest and stops if any checksum differs. Below: all files stacked,
with a `state` column added in memory (the 56th column), and the checksum check.

In [2]:
show(raw_bids)
same = all(hashlib.sha256((ROOT / f).read_bytes()).hexdigest() == h for f, h in zip(files["file"], files["sha256"]))
print(f"manifest: {len(files)} files, {int((manifest['rows'] == 0).sum())} month with no data; checksums match: {same}")

39,958 rows x 56 columns


,report_date,report_begin_date,report_end_date,published_date,office_name,office_city,office_state,market_type,market_location_name,market_location_city,market_location_state,slug_id,slug_name,report_title,grp,...,price Max,price Min Change,price Min Direction,price Max Change,price Max Direction,avg_price,avg_price_year_ago,freight,trans_mode,current,delivery_start,delivery Start Half,delivery_end,delivery End Half,state
0,10/30/2020,10/30/2020,10/30/2020,10/30/2020 14:58:14,"Springfield, IL",Springfield,IL,Point of Sale - Grain,Grain - Illinois,Springfield,IL,3192,AMS_3192,Illinois Grain Bids,Grains,...,10.6125,0.0025,DN,0.0375,UP,10.5654,NaN,Delivered,Truck,No,2021-01-01,NaN,2021-01-01,NaN,IL
1,10/30/2020,10/30/2020,10/30/2020,10/30/2020 14:58:14,"Springfield, IL",Springfield,IL,Point of Sale - Grain,Grain - Illinois,Springfield,IL,3192,AMS_3192,Illinois Grain Bids,Grains,...,10.4450,0.1225,DN,0.1225,DN,10.3717,NaN,Delivered,Truck,Yes,NaN,NaN,NaN,NaN,IL
2,10/30/2020,10/30/2020,10/30/2020,10/30/2020 14:58:14,"Springfield, IL",Springfield,IL,Point of Sale - Grain,Grain - Illinois,Springfield,IL,3192,AMS_3192,Illinois Grain Bids,Grains,...,10.6625,0.0575,UP,0.0175,UP,10.6305,NaN,Delivered,Truck,No,2021-01-01,NaN,2021-01-01,NaN,IL
3,10/30/2020,10/30/2020,10/30/2020,10/30/2020 14:58:14,"Springfield, IL",Springfield,IL,Point of Sale - Grain,Grain - Illinois,Springfield,IL,3192,AMS_3192,Illinois Grain Bids,Grains,...,10.5750,0.0625,DN,0.0475,UP,10.3990,NaN,Delivered,Truck,Yes,NaN,NaN,NaN,NaN,IL
4,10/30/2020,10/30/2020,10/30/2020,10/30/2020 14:58:14,"Springfield, IL",Springfield,IL,Point of Sale - Grain,Grain - Illinois,Springfield,IL,3192,AMS_3192,Illinois Grain Bids,Grains,...,10.9125,0.0475,UP,0.0475,UP,10.6585,NaN,Delivered,Truck,No,2021-01-01,NaN,2021-01-01,NaN,IL


missing per column: {'location_State': 39958, 'location_City': 39958, 'trade_loc': 710, 'class': 16555, 'protein': 32159, 'application': 39958, 'desc': 39853, 'basis_unit': 402, 'basis Min': 402, 'basis Min Futures Month': 402, 'basis Max': 402, 'basis Max Futures Month': 402, 'basis Min Change': 1742, 'basis Min Direction': 1691, 'basis Max Change': 1742, 'basis Max Direction': 1691, 'price Min': 65, 'price Max': 65, 'price Min Change': 476, 'price Min Direction': 425, 'price Max Change': 476, 'price Max Direction': 425, 'avg_price': 65, 'avg_price_year_ago': 11358, 'delivery_start': 26297, 'delivery Start Half': 39341, 'delivery_end': 26297, 'delivery End Half': 39341}
manifest: 71 files, 1 month with no data; checksums match: True


**a2) Data exploration: what is in the raw data?** Three tables:
1. **One row per state:** raw rows, how many are corn, how many of those are for delivery now, how many of those are White
   corn or empty `¢/Bu` rows (what the filters will remove), and the regions and buyer types left.
2. **Raw rows by state × month,** straight from the manifest (0 = no data that month).
3. **Is there a bid on each decision date?** The day of the last report on or before the 15th, per state-month. The gold step
   needs a cash bid on the futures date (the 15th, or the prior trading day: 14 or 13 when the 15th was a weekend).

In [3]:
corn_now = raw_bids[(raw_bids["commodity"] == "Corn") & (raw_bids["current"] == "Yes")]           # corn, deliver now
empty = (corn_now["price_unit"] == "¢/Bu") & corn_now[["avg_price", "price Min", "price Max"]].isna().all(axis=1)
by_state = pd.DataFrame({
    "raw rows": raw_bids.groupby("state").size(),                                                 # all grains
    "corn": raw_bids[raw_bids["commodity"] == "Corn"].groupby("state").size(),                    # corn only
    "corn, deliver now": corn_now.groupby("state").size(),                                        # + deliver now
    "of which White": corn_now[corn_now["class"] == "White"].groupby("state").size(),             # removed by Yellow filter
    "of which empty ¢/Bu": corn_now[empty].groupby("state").size(),                               # removed: no price
    "regions": corn_now.groupby("state")["trade_loc"].nunique(),                                  # distinct regions
    "buyer types": corn_now.groupby("state")["delivery_point"].unique().map(lambda v: ", ".join(sorted(v))),
}).reindex(STATES).fillna(0).astype({"of which White": int, "of which empty ¢/Bu": int})
display(by_state)
display(manifest.pivot(index="month", columns="state", values="rows")[STATES])                   # raw rows per state-month
dates = pd.to_datetime(raw_bids["report_date"], format="%m/%d/%Y")                                # report dates
last_by_15 = dates[dates.dt.day <= 15].groupby([dates.dt.strftime("%Y-%m"), raw_bids["state"]]).max().dt.day
display(last_by_15.unstack()[STATES].astype("Int64"))                                             # blank = no report that month

,raw rows,corn,"corn, deliver now",of which White,of which empty ¢/Bu,regions,buyer types
state,,,,,,,
IL,12486,5786,3032,0,8,12,"Barge Loading Elevators, Country Elevators, Mi..."
IA,3750,1771,1771,0,0,6,"Country Elevators, Terminal Elevators"
NE,5598,1750,1750,0,0,7,Country Elevators
SD,5383,1256,1256,0,0,5,Country Elevators
MN,639,319,255,0,0,1,Country Elevators
IN,12102,4722,2026,147,0,9,"Barge Loading Elevators, Country Elevators, Te..."


state,IL,IA,NE,SD,MN,IN
month,,,,,,
2020-10,1008,308,461,483,44,0
2021-04,1029,294,451,456,42,388
2021-10,976,294,424,462,43,359
2022-04,994,300,440,440,40,341
2022-10,985,300,440,440,42,1180
2023-04,949,285,399,417,38,1583
2023-10,1101,315,462,461,44,1197
2024-04,1102,330,484,441,44,1923
2024-10,1115,330,484,483,46,1345


state,IL,IA,NE,SD,MN,IN
report_date,,,,,,
2020-10,15,15,15,15,15,<NA>
2021-04,15,15,15,15,15,13
2021-10,15,15,15,15,15,12
2022-04,14,14,14,14,14,13
2022-10,14,14,14,14,14,14
2023-04,14,14,14,14,14,14
2023-10,13,13,13,13,13,13
2024-04,15,15,15,15,15,15
2024-10,15,15,15,15,15,15


**b) Cleaning code.** `03_build_silver.py` **lines 26-42** (column lists) and **lines 57-92** (filter, rename, convert, check).

- **Columns:** keep 14 of the 56: `state` plus the 13 used before (the rest are city, protein, basis change, delivery window, ...).
- **Rows:** corn only; deliver now only (the question is selling *now* versus later); **Yellow corn only** (Indiana also
  reports White corn, a different product; futures are for Yellow #2); then drop the 8 empty Illinois `¢/Bu` rows (no price,
  and each repeats the key of a real `$ Per Bushel` row, so no price is lost).
- **Renames and types:** USDA names → project names; `Yes`/`No` → `True`/`False`; `MM/DD/YYYY` text → real date.
- **Primary key:** (`date`, `state`, `region`, `buyer_type`, `transport_mode`) must be unique. `state` is needed because region
  names repeat across states (Iowa and South Dakota also have a North Central); `transport_mode` because Indiana's Ohio River
  barge elevators post a Truck and a Truck/Rail bid on the same day.

In [4]:
KEEP = ["report_date", "state", "commodity", "trade_loc", "delivery_point", "current",           # 03 lines 26-28:
        "avg_price", "price Min", "price Max", "price_unit",                                     # 14 columns kept
        "class", "grade", "freight", "trans_mode"]
RENAME = {"report_date": "date", "trade_loc": "region", "delivery_point": "buyer_type",          # 03 lines 29-41:
          "current": "deliver_now", "avg_price": "cash_price", "price Min": "cash_price_low",    # USDA name -> project name
          "price Max": "cash_price_high", "class": "grain_class", "grade": "grain_grade",
          "freight": "price_point", "trans_mode": "transport_mode"}
PRICE_COLS = ["cash_price", "cash_price_low", "cash_price_high"]                                # 03 line 42

df = raw_bids[KEEP].copy()                                                   # line 57: drop 42 of 56 columns
print(f"columns kept: {raw_bids.shape[1]} → {df.shape[1]}")
n = len(df); df = df[df["commodity"] == "Corn"]                              # line 63: corn only
print(f"corn only: {n:,} → {len(df):,}")                                     # soybeans and wheat removed
n = len(df); df = df[df["current"] == "Yes"]                                 # line 64: deliver now only
print(f"deliver now only: {n:,} → {len(df):,}")                              # forward bids removed
n = len(df); df = df[df["class"] == "Yellow"]                                # line 65: Yellow corn only
print(f"Yellow corn only: {n:,} → {len(df):,}")                              # Indiana White corn removed
df = df.rename(columns=RENAME)                                               # line 68: project names
df["deliver_now"] = df["deliver_now"].map({"Yes": True, "No": False})        # line 69: Yes/No -> True/False
df["date"] = pd.to_datetime(df["date"], format="%m/%d/%Y")                   # line 72: MM/DD/YYYY -> real date
empty_cents = (df["price_unit"] == "¢/Bu") & df[PRICE_COLS].isna().all(axis=1)   # line 76: ¢/Bu rows with no price
assert empty_cents.sum() == 8                                                # line 77: exactly the 8 known rows
n = len(df); df = df[~empty_cents]                                           # line 78: drop them
print(f"drop empty ¢/Bu rows: {n:,} → {len(df):,}")
PK = ["date", "state", "region", "buyer_type", "transport_mode"]             # line 83: primary key
assert df.duplicated(PK).sum() == 0 and df[PK].notna().all().all()          # lines 84-91: unique, no blanks
bids = df.sort_values(PK).reset_index(drop=True)                             # line 92: sorted by the key

columns kept: 56 → 14
corn only: 39,958 → 15,604
deliver now only: 15,604 → 10,090
Yellow corn only: 10,090 → 9,943
drop empty ¢/Bu rows: 9,943 → 9,935


**c) Clean table.** One row = one corn bid from one buyer type, in one region of one state, by one transport mode, on one day
(`data/clean/corn_bids_current.csv`). The second table counts rows by state.

In [5]:
show(clean_bids)
display(clean_bids["state"].value_counts().reindex(STATES).rename("rows").to_frame().T)   # rows by state
summary(raw_bids, clean_bids, bids.to_csv(index=False, na_rep="MISSING", date_format="%Y-%m-%d") == text("data/clean/corn_bids_current.csv"))

9,935 rows x 14 columns


,date,state,commodity,region,buyer_type,deliver_now,cash_price,cash_price_low,cash_price_high,price_unit,grain_class,grain_grade,price_point,transport_mode
0,2020-10-01,IA,Corn,North Central,Country Elevators,True,3.5305,3.4475,3.5775,$ Per Bushel,Yellow,US #2,Delivered,Truck
1,2020-10-01,IA,Corn,Northeast,Country Elevators,True,3.5364,3.4175,3.6675,$ Per Bushel,Yellow,US #2,Delivered,Truck
2,2020-10-01,IA,Corn,Northwest,Country Elevators,True,3.5318,3.4775,3.5575,$ Per Bushel,Yellow,US #2,Delivered,Truck
3,2020-10-01,IA,Corn,South Central,Country Elevators,True,3.4975,3.4475,3.6775,$ Per Bushel,Yellow,US #2,Delivered,Truck
4,2020-10-01,IA,Corn,Southeast,Country Elevators,True,3.5675,3.4875,3.6975,$ Per Bushel,Yellow,US #2,Delivered,Truck


missing per column: none


state,IL,IA,NE,SD,MN,IN
rows,3024,1771,1750,1256,255,1879


rows 39,958 → 9,935, columns 56 → 14   (code above reproduces the saved file exactly: True)


**c2) Data exploration: what is left after cleaning?** Each descriptive column now has a single value, except
`transport_mode` (some Indiana bids are `Truck/Rail`). The second table is the mean cash price ($/bu) per state and month.

In [6]:
COLS = ["state", "region", "buyer_type", "deliver_now", "price_unit", "grain_class", "grain_grade",   # descriptive columns
        "price_point", "transport_mode", "cash_price"]                                                # and the price
display(explore(clean_bids, COLS))                                                                   # each column at a glance
month = clean_bids["date"].str[:7]                                                                   # pull month, e.g. 2020-10
display(clean_bids.pivot_table(index=month, columns="state", values="cash_price", aggfunc="mean")[STATES].round(2))

,type,distinct values,missing,most common (rows)
state,object,6,0,"IL (3,024), IN (1,879), IA (1,771)"
region,object,19,0,"Southwest (1,187), Northeast (935), Central (929)"
buyer_type,object,4,0,"Country Elevators (7,910), Barge Loading Eleva..."
deliver_now,bool,1,0,"True (9,935)"
price_unit,object,1,0,"$ Per Bushel (9,935)"
grain_class,object,1,0,"Yellow (9,935)"
grain_grade,object,1,0,"US #2 (9,935)"
price_point,object,1,0,"Delivered (9,935)"
transport_mode,object,2,0,"Truck (9,578), Truck/Rail (357)"
cash_price,float64,6113,0,"4.49 (15), 3.8875 (14), 3.9175 (14)"


state,IL,IA,NE,SD,MN,IN
date,,,,,,
2020-10,3.85,3.73,3.74,3.47,3.60,NaN
2021-04,6.16,5.99,5.95,6.01,5.93,6.14
2021-10,5.15,5.16,5.19,5.33,5.25,5.15
2022-04,7.75,7.66,7.61,7.58,7.66,7.73
2022-10,6.55,6.72,7.21,6.54,6.73,6.40
2023-04,6.55,6.62,6.70,6.51,6.64,6.48
2023-10,4.64,4.62,4.63,4.31,4.62,4.56
2024-04,4.23,4.35,4.21,4.02,4.21,4.20
2024-10,3.84,3.85,3.95,3.61,4.02,3.76


**Rows removed:** 24,354 not corn, 5,514 corn for later delivery, 147 White corn (Indiana), 8 empty `¢/Bu` rows (Illinois)
= 30,023. Illinois alone: 12,486 → 3,024, the same rows as before the other states were added.

**What the exploration shows:**
- **Indiana Oct 2020 has no data** (the API returned nothing), and Indiana's Apr 2021, Oct 2021 and Apr 2022 are **weekly**: the
  last report before the 15th is on the 13th or 12th, not the futures date. Every other state-month has a report on the decision date.
- **Minnesota has only one region** (South), and from Apr 2025 its report also lists forward bids, which doubles its raw rows
  (the deliver-now rows stay about 21-23 a month).
- **Nebraska and South Dakota** report Country Elevators only. Some raw rows in Iowa and Indiana have no region, but they are all
  soybeans or wheat, so no corn row is affected.
- **For now, only Illinois is used after this step** (Part 3).

*Details:* [processing log: 2026-11-21: cash bids for 6 states in the pipeline (03, 06, 07 changed; rerun 03 → 09)](../docs/processing_log.md#2026-11-21-cash-bids-for-6-states-in-the-pipeline-03-06-07-changed-rerun-03--09), [processing log: 2026-11-21: USDA grain bids pulled for 5 more states (raw only, no cleaning)](../docs/processing_log.md#2026-11-21-usda-grain-bids-pulled-for-5-more-states-raw-only-no-cleaning) and [processing log: 03_build_silver.py: cash bids (silver table)](../docs/processing_log.md#03_build_silverpy-cash-bids-silver-table)

## 1.2 May futures (`04_clean_futures.py`)

**a) Raw table.** 12 settlement prices of the CBOT May corn contract, typed by hand from Barchart exactly as quoted,
in **cents and eighths**: for each decision year, one on the October date (sell futures = open the hedge) and one on the
April date (buy back = close the hedge), always the **same** May contract.

In [7]:
show(raw_futures)

12 rows x 5 columns


,decision_year,date,contract,settle_raw,source
0,2020,2020-10-15,ZCK21,409-2,Barchart settlement
1,2020,2021-04-15,ZCK21,590-0,Barchart settlement
2,2021,2021-10-15,ZCK22,538-6,Barchart settlement
3,2021,2022-04-14,ZCK22,790-2,Barchart settlement
4,2022,2022-10-14,ZCK23,697-2,Barchart settlement


missing per column: none


**b) Cleaning code.** `04_clean_futures.py` **lines 26-61**.

- **Rows:** none removed (all 12 are needed).
- **Rename:** `contract` → `futures_contract`. **New columns:** `contract_name` (the code in words), `leg` (Oct = open
  hedge, Apr = close hedge, from the date's month) and `may_futures_price`.
- **Conversion:** `583-2` means 583 and **2/8** cents = 583.25¢ = **$5.8325**. Reading it as 583.2 would be wrong, so the
  format is checked strictly (a dash, then an eighth 0-7). Not rounded: odd eighths need 5 decimals (`583-1` = $5.83125).
  A blank quote would stay blank (MISSING); there are none.

In [8]:
n = len(raw_futures)                                                          # rows before
df = raw_futures.rename(columns={"contract": "futures_contract"})             # line 27: rename
def contract_name(code):                                                      # lines 31-35: "ZCK21" -> "May 2021 corn"
    m = re.fullmatch(r"ZC([A-Z])(\d{2})", code)                                # ZC = corn, K = May, 21 = 2021
    if not m or m.group(1) != "K":                                            # only May contracts expected
        raise ValueError(f"Unexpected contract code: {code}")                 # anything else stops
    return f"May 20{m.group(2)} corn"                                         # the code in words
df["contract_name"] = df["futures_contract"].map(contract_name)               # line 36: new column
LEGS = {10: "Oct (open hedge)", 4: "Apr (close hedge)"}                      # line 39: month -> hedge leg
df["leg"] = pd.to_datetime(df["date"]).dt.month.map(LEGS)                     # lines 40-43: new column
def to_dollars(raw):                                                          # lines 48-56: cents and eighths -> $/bu
    if raw.strip() == "":                                                     # blank quote
        return None                                                           # stays MISSING, never guessed
    m = re.fullmatch(r"(\d+)-([0-7])", raw.strip())                            # strict format: 583-2, eighths 0-7
    if not m:                                                                 # anything else
        raise ValueError(f"Unexpected settle_raw format: {raw!r}")            # stops the script
    return (int(m.group(1)) + int(m.group(2)) / 8) / 100                      # 583 + 2/8 = 583.25¢ -> $5.8325
df["may_futures_price"] = df["settle_raw"].map(to_dollars).astype(float)      # line 57: new price column
futures = df[["decision_year", "date", "futures_contract", "contract_name",   # lines 60-61: fixed column order
              "leg", "settle_raw", "source", "may_futures_price"]]
print(f"rows: {n} → {len(futures)} (none removed);  583-2 → {to_dollars('583-2')},  583-1 → {to_dollars('583-1')}")

rows: 12 → 12 (none removed);  583-2 → 5.8325,  583-1 → 5.83125


**c) Clean table.** Same 12 rows with the price in $/bu (`data/clean/may_futures.csv`). Four dates are not the 15th
(2022-04-14, 2022-10-14, 2023-04-14, 2023-10-13): when the 15th was not a trading day, the nearest **prior** trading day
was used, so no later price is used.

In [9]:
show(clean_futures)
summary(raw_futures, clean_futures, futures.to_csv(index=False) == text("data/clean/may_futures.csv"))

12 rows x 8 columns


,decision_year,date,futures_contract,contract_name,leg,settle_raw,source,may_futures_price
0,2020,2020-10-15,ZCK21,May 2021 corn,Oct (open hedge),409-2,Barchart settlement,4.0925
1,2020,2021-04-15,ZCK21,May 2021 corn,Apr (close hedge),590-0,Barchart settlement,5.9000
2,2021,2021-10-15,ZCK22,May 2022 corn,Oct (open hedge),538-6,Barchart settlement,5.3875
3,2021,2022-04-14,ZCK22,May 2022 corn,Apr (close hedge),790-2,Barchart settlement,7.9025
4,2022,2022-10-14,ZCK23,May 2023 corn,Oct (open hedge),697-2,Barchart settlement,6.9725


missing per column: none
rows 12 → 12, columns 5 → 8   (code above reproduces the saved file exactly: True)


*Details:* [processing log: 04_clean_futures.py: May futures](../docs/processing_log.md#04_clean_futurespy-may-futures)

## 1.3 T-bill rate (`05_tbill_rates.py`)

**a) Raw table.** FRED series DTB6: the daily 6-month T-bill rate in **percent**, from 1958 on. Bond-market holidays
are blank rows.

In [10]:
show(raw_tbill)

17,707 rows x 2 columns


,observation_date,DTB6
0,1958-12-09,3.09
1,1958-12-10,3.07
2,1958-12-11,3.06
3,1958-12-12,3.04
4,1958-12-15,3.03


missing per column: {'DTB6': 755}


**b) Cleaning code.** `05_tbill_rates.py` **lines 24-51**.

- **Rows:** drop the blank holiday rows (skipped, never filled); then, for each of the 6 decision dates D, keep only the
  10 calendar days **D-9 to D** (D included) and average them, so 17,707 daily rows become 6.
- **Why:** the rate prices the T-bill interest given up by holding corn instead of cash; a 10-day average smooths one-day
  jumps. Only dates up to D are used (**no future leakage**); the farmer can buy a T-bill at the market rate on D itself.
- **Types:** text → date and number; percent → **decimal** (`4.02` → `0.0402`) for calculations, plus a `4.02%` text copy for reading.
- **Shortened:** the MISSING branch for an empty window (never happens) and the single-day comparison print are left out.

In [11]:
DECISION_DATES = ["2020-10-15", "2021-10-15", "2022-10-14",                   # lines 24-25: the Oct dates of may_futures
                  "2023-10-13", "2024-10-15", "2025-10-15"]                   # (prior trading day where needed)
fred = raw_tbill.copy()                                                       # line 28: read as text
fred["observation_date"] = pd.to_datetime(fred["observation_date"])           # line 29: text -> date
n = len(fred); fred = fred[fred["DTB6"].str.strip() != ""].copy()             # line 30: drop blank holiday rows
print(f"drop blank holiday rows: {n:,} → {len(fred):,}")                      # skipped, not filled
fred["DTB6"] = fred["DTB6"].astype(float)                                     # line 31: text -> number (percent)
rows = []                                                                     # one result row per decision date
for d in pd.to_datetime(DECISION_DATES):                                      # line 35: each decision date D
    start = d - pd.Timedelta(days=9)                                          # line 36: window D-9 .. D
    win = fred[(fred["observation_date"] >= start) & (fred["observation_date"] <= d)]   # line 37: rows in the window
    assert win["observation_date"].max() <= d                                 # line 38: no date after D
    avg = win["DTB6"].mean()                                                  # line 39: average of published days
    rows.append({"decision_year": d.year, "decision_date": d.strftime("%Y-%m-%d"),      # lines 40-50: one row
                 "window_start": start.strftime("%Y-%m-%d"), "window_end": d.strftime("%Y-%m-%d"),
                 "n_days_used": len(win),                                     # weekends/holidays have no rate
                 "dtb6_avg_pct": f"{avg:.2f}%",                               # text, for reading
                 "rate_decimal": avg / 100})                                  # percent -> decimal
tbill = pd.DataFrame(rows)                                                    # line 51
assert not tbill["decision_date"].duplicated().any()                          # line 54: primary key unique
print(f"10-day windows ending on the 6 decision dates: {len(fred):,} → {len(tbill)}")

drop blank holiday rows: 17,707 → 16,952
10-day windows ending on the 6 decision dates: 16,952 → 6


**c) Clean table.** One row per decision date (`data/clean/tbill_decision.csv`). 2024 used only 6 days because 2024-10-14
is blank in FRED.

In [12]:
show(clean_tbill)
summary(raw_tbill, clean_tbill, tbill.to_csv(index=False) == text("data/clean/tbill_decision.csv"))

6 rows x 7 columns


,decision_year,decision_date,window_start,window_end,n_days_used,dtb6_avg_pct,rate_decimal
0,2020,2020-10-15,2020-10-06,2020-10-15,7,0.12%,0.001186
1,2021,2021-10-15,2021-10-06,2021-10-15,7,0.06%,0.000614
2,2022,2022-10-14,2022-10-05,2022-10-14,7,4.02%,0.040229
3,2023,2023-10-13,2023-10-04,2023-10-13,7,5.33%,0.053343
4,2024,2024-10-15,2024-10-06,2024-10-15,6,4.30%,0.042983


missing per column: none
rows 17,707 → 6, columns 2 → 7   (code above reproduces the saved file exactly: True)


*Details:* [processing log: 05_tbill_rates.py: T-bill rate for the hurdle](../docs/processing_log.md#05_tbill_ratespy-t-bill-rate-for-the-hurdle)

## 1.4 Storage costs (`data/raw/storage_costs.csv`, no cleaning script)

The cost of physically storing a bushel from Oct 15 to Apr 15, typed by hand from articles (user-approved 2026-11-13),
in **three cases**:

| Case | What it represents | Source | Cents/bu, 6 months |
|---|---|---|---|
| **low** | on-farm bins | Iowa Farm Bureau (2016, figure not attributed): 1-2¢/bu/month, midpoint 1.5¢ × 6. **Weaker source**, same every year | 9 |
| **high** | commercial elevator | farmdoc daily, Zulauf, Paulson & Janzen (Oct 2, 2025): "physical storage" $/bu/year × 0.5 (one Ohio country elevator) | 35.5 to 65 |
| **mid** | in between | (low + high) / 2 | 22.25 to 37 |

Only physical storage is used, not total cost, because interest is already counted from the T-bill rate.

**a) Raw table.** One row per decision year.

In [13]:
show(raw_storage)

6 rows x 7 columns


,decision_year,storage_low_cents,storage_mid_cents,storage_high_cents,source_url_low,source_url_high,note
0,2020,9.0,22.25,35.5,https://www.iowafarmbureau.com/Article/Keep-a-...,https://farmdocdaily.illinois.edu/2025/10/annu...,low: typed from article (on-farm 1.5 c/bu/mont...
1,2021,9.0,22.50,36.0,https://www.iowafarmbureau.com/Article/Keep-a-...,https://farmdocdaily.illinois.edu/2025/10/annu...,low: typed from article (on-farm 1.5 c/bu/mont...
2,2022,9.0,22.50,36.0,https://www.iowafarmbureau.com/Article/Keep-a-...,https://farmdocdaily.illinois.edu/2025/10/annu...,low: typed from article (on-farm 1.5 c/bu/mont...
3,2023,9.0,37.00,65.0,https://www.iowafarmbureau.com/Article/Keep-a-...,https://farmdocdaily.illinois.edu/2025/10/annu...,low: typed from article (on-farm 1.5 c/bu/mont...
4,2024,9.0,37.00,65.0,https://www.iowafarmbureau.com/Article/Keep-a-...,https://farmoffice.osu.edu/blog/mon-10062025-4...,low: typed from article (on-farm 1.5 c/bu/mont...


missing per column: none


**b) Checking code.** `07_build_gold.py` **lines 72-78**. There is no cleaning step: the file is already in cents for the
6-month hold, so `07` reads it directly, runs these checks, and keeps only the year and the three cost columns.

- **Why the checks:** exactly one row per year 2020-2025 (so the join in Part 3 cannot duplicate or lose a year), and
  `mid` must really be the average of `low` and `high` (catches a typing error).

In [14]:
storage_cols = [f"storage_{c}_cents" for c in CASES]                          # line 73: the 3 cost columns
assert sorted(raw_storage["decision_year"]) == list(range(2020, 2026))        # line 74: one row per year 2020-2025
assert not raw_storage["decision_year"].duplicated().any()                    # line 75: no duplicate year
assert ((raw_storage["storage_low_cents"] + raw_storage["storage_high_cents"]) / 2   # lines 76-77: mid = (low + high) / 2
        - raw_storage["storage_mid_cents"]).abs().max() < 1e-9
storage_used = raw_storage[["decision_year"] + storage_cols]                  # line 78: columns copied into gold
print(f"columns kept: {raw_storage.shape[1]} → {storage_used.shape[1]} (source links and notes stay in the raw file)")
print("2025 note:", raw_storage.loc[raw_storage["decision_year"] == 2025, "note"].item())

columns kept: 7 → 4 (source links and notes stay in the raw file)
2025 note: low: typed from article (on-farm 1.5 c/bu/month x 6; weaker source, Iowa Farm Bureau 2016, unattributed). high: FLAG no 2025 value published; 2024 physical storage 1.30 $/bu/yr carried forward x 0.5. mid = (low + high) / 2


**c) What `07` uses.** No clean file is saved; these four columns go straight into the gold table.

In [15]:
show(storage_used)
summary(raw_storage, storage_used)

6 rows x 4 columns


,decision_year,storage_low_cents,storage_mid_cents,storage_high_cents
0,2020,9.0,22.25,35.5
1,2021,9.0,22.50,36.0
2,2022,9.0,22.50,36.0
3,2023,9.0,37.00,65.0
4,2024,9.0,37.00,65.0


missing per column: none
rows 6 → 6, columns 7 → 4


> **Flag:** the **2025 high** value (65¢) is the 2024 value **carried forward**, because farmdoc had not published a
> 2025 figure. It is a placeholder, not a measured number.

Known limits: halving the yearly commercial cost probably **understates** it (elevators often charge a flat fee and then
a monthly rate; about 68% of the yearly cost under 2014 central-Illinois rates), and the rate is from Ohio, not Illinois.
*Details:* [processing log: 2026-11-13: storage cost cases added to gold (06 → 07 → 08 rerun)](../docs/processing_log.md#2026-11-13-storage-cost-cases-added-to-gold-06--07--08-rerun)

# Part 2: Database (`06_build_db.py`)

`06` loads the three clean CSVs into one SQLite file, `data/clean/corn.db`, and adds a `calendar` table: every day
from 2020-01-01 to 2026-12-31.

- A **primary key (PK)** is the column(s) that identify a row, so they can never repeat: in `corn_bids` it is
  (`date`, `state`, `region`, `buyer_type`, `transport_mode`), so two different prices for Illinois North Central country
  elevators by truck on the same day cannot both exist.
- A **foreign key (FK)** says a value must already exist in another table: every `date` in `corn_bids` must be a real
  day in `calendar`, so a typo like `2022-02-30` is rejected instead of silently joining to nothing.

**a) Tables in the database**, with their row counts, PKs and FKs, read from SQLite itself (`gold_yearly` is added later by `07`).

In [16]:
show(db_tables)

5 rows x 4 columns


,table,rows,primary key,foreign keys
0,calendar,2557,date,none
1,corn_bids,9935,"date, state, region, buyer_type, transport_mode",date → calendar(date)
2,may_futures,12,date,date → calendar(date)
3,tbill_decision,6,decision_date,decision_date → calendar(date)
4,gold_yearly,12,"decision_year, region",apr_date → calendar(date); oct_date → calendar...


missing per column: none


**b) Checking code.** `06_build_db.py` **lines 98-123** (checks before loading) and **lines 146-147** (SQLite's own check
after loading).

- **What:** build the calendar; then for each clean CSV count duplicate PKs, blank PK values and FK dates that are not in
  the calendar. All must be 0, otherwise `06` prints the problem rows, stops and saves nothing.
- **Why:** checking *before* loading means a bad file never reaches the database. SQLite's `foreign_key_check` is a second,
  independent check after loading. Foreign keys are **off by default** in SQLite; `06` turns them on with `PRAGMA foreign_keys = ON`.
- The CSVs are re-read **as text** here, as in `06`, so a date is compared exactly as written.

In [17]:
calendar = set(pd.date_range("2020-01-01", "2026-12-31", freq="D").strftime("%Y-%m-%d"))   # lines 98-99: every day
KEYS = {"corn_bids": ("corn_bids_current", ["date", "state", "region", "buyer_type", "transport_mode"], "date"),   # lines 72-81
        "may_futures": ("may_futures", ["date"], "date"),
        "tbill_decision": ("tbill_decision", ["decision_date"], "decision_date")}
checks = []                                                                   # one result row per table
for t, (csv, pk, fk) in KEYS.items():                                         # line 108: each table
    df = pd.read_csv(ROOT / f"data/clean/{csv}.csv", dtype=str, keep_default_na=False)   # line 102: read as text
    checks.append({"table": t, "rows": len(df),
                   "duplicate PKs": int(df.duplicated(pk, keep=False).sum()),          # line 110: must be 0
                   "blank PK values": int((df[pk] == "").any(axis=1).sum()),           # line 113: must be 0
                   "FK dates not in calendar": int((~df[fk].isin(calendar)).sum())})   # line 116: must be 0
checks = pd.DataFrame(checks)
fk_violations = con.execute("PRAGMA foreign_key_check").fetchall()           # line 147: [] = no violations
print("calendar rows:", len(calendar), "  SQLite foreign_key_check:", fk_violations)

calendar rows: 2557   SQLite foreign_key_check: []


**c) Check results.** Every count must be 0.

In [18]:
show(checks)
print("all checks passed:", int(checks.iloc[:, 2:].to_numpy().sum()) == 0 and fk_violations == [])

3 rows x 5 columns


,table,rows,duplicate PKs,blank PK values,FK dates not in calendar
0,corn_bids,9935,0,0,0
1,may_futures,12,0,0,0
2,tbill_decision,6,0,0,0


missing per column: none
all checks passed: True


Row counts match the log (calendar 2,557, corn_bids 9,935 for 6 states, may_futures 12, tbill_decision 6).
*Details:* [processing log: 06_build_db.py: SQLite database](../docs/processing_log.md#06_build_dbpy-sqlite-database)

# Part 3: Gold table (`07_build_gold.py`)

The gold table holds everything needed to judge "store or sell": **one row per region per decision year**, for
**Country Elevators** (the buyers a farmer actually delivers to) in **North Central** and **South Central** Illinois,
decision years 2020-2025: 2 × 6 = 12 rows. **For now it uses Illinois only**: the other five states are in the clean cash
bids (1.1) but not yet in the analysis. The `state = 'IL'` filter matters because Iowa and South Dakota also have regions named
North Central and South Central.

**a) Main input: the clean cash bids** (table `corn_bids` in the database, 6 states). The gold step picks 24 of these rows
(Illinois, 2 regions × 12 hedge dates) and adds the futures price, the T-bill rate and the storage costs.

In [19]:
show(clean_bids)

9,935 rows x 14 columns


,date,state,commodity,region,buyer_type,deliver_now,cash_price,cash_price_low,cash_price_high,price_unit,grain_class,grain_grade,price_point,transport_mode
0,2020-10-01,IA,Corn,North Central,Country Elevators,True,3.5305,3.4475,3.5775,$ Per Bushel,Yellow,US #2,Delivered,Truck
1,2020-10-01,IA,Corn,Northeast,Country Elevators,True,3.5364,3.4175,3.6675,$ Per Bushel,Yellow,US #2,Delivered,Truck
2,2020-10-01,IA,Corn,Northwest,Country Elevators,True,3.5318,3.4775,3.5575,$ Per Bushel,Yellow,US #2,Delivered,Truck
3,2020-10-01,IA,Corn,South Central,Country Elevators,True,3.4975,3.4475,3.6775,$ Per Bushel,Yellow,US #2,Delivered,Truck
4,2020-10-01,IA,Corn,Southeast,Country Elevators,True,3.5675,3.4875,3.6975,$ Per Bushel,Yellow,US #2,Delivered,Truck


missing per column: none


**b) Joining and calculating code.** `07_build_gold.py` **lines 47-79** (joins) and **lines 102-115** (calculations), all in cents/bu.

- **Join 1 (lines 47-63):** for each futures row of one leg (Oct or Apr), take both regions and look up the Illinois Country
  Elevators cash bid on the **same date**. A `LEFT JOIN` keeps the row even if a bid is missing (it would show as blank
  and stop the script), instead of silently dropping it. The first lines (not in the script) only count which bids the join picks.
- **Join 2 (line 64):** Oct and Apr legs side by side, matched on (`decision_year`, `region`).
- **Joins 3-4 (lines 66-79):** the rate on the **Oct date** (when the money would go into T-bills), and the storage costs by year.
- **Calculations (lines 102-115):** the gap is **futures − cash**, so a wide gap means a weak local price. The hedge locks
  in futures, so storing earns how much that gap **narrows** from October to April:

| Column | Formula | Meaning |
|---|---|---|
| `gap_oct_cents` | (oct_futures − oct_cash) × 100 | how far cash sits below futures in October (**X**) |
| `gap_apr_cents` | (apr_futures − apr_cash) × 100 | the same gap in April |
| `gap_closed_cents` | gap_oct − gap_apr | what the hedged storage earns (**Y** = the README's Target, Basis(Apr) − Basis(Oct)) |
| `interest_cents` | rate_decimal × 0.5 × oct_cash × 100 | T-bill interest given up for 6 months on the October cash price |
| `breakeven_storage_cents` | gap_closed − interest | the most storage could cost and storing still breaks even |
| `hurdle_{c}_cents` | storage_c + interest | what storing must earn, for case c = low, mid, high |
| `net_gain_{c}_cents` | gap_closed − storage_c − interest | gain over selling at harvest (selling = 0) |
| `store_wins_{c}` | net_gain_c > 0 | True if storing beat selling |

In [20]:
STATE, REGIONS, BUYER_TYPE = "IL", ("North Central", "South Central"), "Country Elevators"   # lines 37-39
n = len(clean_bids); b = clean_bids[clean_bids["date"].isin(clean_futures["date"])]   # count only: bids on the 12 hedge dates
print(f"bids on the 12 hedge dates: {n:,} → {len(b):,}")
n = len(b); b = b[b["state"] == STATE]                                                # count only: Illinois
print(f"Illinois only: {n:,} → {len(b):,}")
n = len(b); b = b[(b["buyer_type"] == BUYER_TYPE) & b["region"].isin(REGIONS)]        # count only: 2 regions, country elevators
print(f"North/South Central country elevators: {n:,} → {len(b):,}")
LEG_SQL = """
SELECT f.decision_year, r.region, f.date, b.cash_price, f.may_futures_price
FROM may_futures f
CROSS JOIN (SELECT ? AS region UNION ALL SELECT ?) r      -- every futures row x both regions
LEFT JOIN corn_bids b                                      -- cash bid on the same day, if any
       ON b.date = f.date AND b.state = ? AND b.region = r.region AND b.buyer_type = ?
WHERE f.leg = ?                                            -- one leg at a time
"""
def leg(name, prefix):                                                         # lines 57-60: one leg, renamed
    df = pd.read_sql(LEG_SQL, con, params=(*REGIONS, STATE, BUYER_TYPE, name)) # run the join
    return df.rename(columns={"date": f"{prefix}_date", "cash_price": f"{prefix}_cash",
                              "may_futures_price": f"{prefix}_futures"})       # oct_cash, apr_futures, ...
oct_, apr_ = leg("Oct (open hedge)", "oct"), leg("Apr (close hedge)", "apr")    # lines 62-63
g = oct_.merge(apr_, on=["decision_year", "region"], how="outer")              # line 64: Oct + Apr side by side
print(f"join 1-2: Oct leg {len(oct_)} rows + Apr leg {len(apr_)} rows → {len(g)} rows")
rate = pd.read_sql("SELECT decision_date AS oct_date, rate_decimal FROM tbill_decision", con)   # line 67
g = g.merge(rate, on="oct_date", how="left")                                   # line 68: rate on the decision date
g.insert(2, "buyer_type", BUYER_TYPE)                                          # line 69
g = g.merge(storage_used, on="decision_year", how="left")                      # line 78: storage cases by year
g = g.sort_values(["decision_year", "region"]).reset_index(drop=True)          # line 79
print(f"join 3-4 (rate, storage): {len(g)} rows, {g.shape[1]} columns, missing values: {int(g.isna().sum().sum())}")
g["gap_oct_cents"] = (g["oct_futures"] - g["oct_cash"]) * 100                  # line 103: X, Oct gap
g["gap_apr_cents"] = (g["apr_futures"] - g["apr_cash"]) * 100                  # line 104: Apr gap
g["gap_closed_cents"] = g["gap_oct_cents"] - g["gap_apr_cents"]                # line 105: Y, how much it narrowed
g["interest_cents"] = g["rate_decimal"] * 0.5 * g["oct_cash"] * 100            # line 107: 6 months of T-bill interest
g["breakeven_storage_cents"] = g["gap_closed_cents"] - g["interest_cents"]     # line 109: max storage to break even
for c in CASES:                                                                # lines 112-115: each storage case
    g[f"hurdle_{c}_cents"] = g[f"storage_{c}_cents"] + g["interest_cents"]     # what storing must earn
    g[f"net_gain_{c}_cents"] = g["gap_closed_cents"] - g[f"storage_{c}_cents"] - g["interest_cents"]   # gain vs selling
    g[f"store_wins_{c}"] = g[f"net_gain_{c}_cents"] > 0                        # storing beat selling?
g = g[gold.columns]                                                            # line 158: saved column order
assert not g.duplicated(["decision_year", "region"]).any()                     # PK (decision_year, region) unique
print(f"calculated columns: {len(g)} rows, {g.shape[1]} columns")

bids on the 12 hedge dates: 9,935 → 467
Illinois only: 467 → 144
North/South Central country elevators: 144 → 24
join 1-2: Oct leg 12 rows + Apr leg 12 rows → 12 rows
join 3-4 (rate, storage): 12 rows, 13 columns, missing values: 0
calculated columns: 12 rows, 27 columns


**c) Gold table** (`data/gold/gold_yearly.csv`, also table `gold_yearly` in the database). PK (`decision_year`, `region`)
is unique. Full column meanings: [data dictionary](../docs/data_dictionary.md#gold_yearly).

In [21]:
show(gold)
summary(clean_bids, gold, g.to_csv(index=False, na_rep="MISSING") == text("data/gold/gold_yearly.csv"))

12 rows x 27 columns


,decision_year,region,buyer_type,oct_date,oct_cash,oct_futures,apr_date,apr_cash,apr_futures,rate_decimal,gap_oct_cents,gap_apr_cents,gap_closed_cents,interest_cents,breakeven_storage_cents,storage_low_cents,storage_mid_cents,storage_high_cents,hurdle_low_cents,hurdle_mid_cents,hurdle_high_cents,net_gain_low_cents,net_gain_mid_cents,net_gain_high_cents,store_wins_low,store_wins_mid,store_wins_high
0,2020,North Central,Country Elevators,2020-10-15,3.7945,4.0925,2021-04-15,5.8211,5.9000,0.001186,29.80,7.89,21.91,0.224960,21.685040,9.0,22.25,35.5,9.224960,22.474960,35.724960,12.685040,-0.564960,-13.814960,True,False,False
1,2020,South Central,Country Elevators,2020-10-15,3.7987,4.0925,2021-04-15,5.9019,5.9000,0.001186,29.38,-0.19,29.57,0.225209,29.344791,9.0,22.25,35.5,9.225209,22.475209,35.725209,20.344791,7.094791,-6.155209,True,True,False
2,2021,North Central,Country Elevators,2021-10-15,4.9819,5.3875,2022-04-14,7.6969,7.9025,0.000614,40.56,20.56,20.00,0.153015,19.846985,9.0,22.50,36.0,9.153016,22.653015,36.153016,10.846985,-2.653015,-16.153015,True,False,False
3,2021,South Central,Country Elevators,2021-10-15,5.0663,5.3875,2022-04-14,7.8063,7.9025,0.000614,32.12,9.62,22.50,0.155608,22.344392,9.0,22.50,36.0,9.155608,22.655608,36.155608,13.344392,-0.155608,-13.655608,True,False,False
4,2022,North Central,Country Elevators,2022-10-14,6.6150,6.9725,2023-04-14,6.5856,6.6625,0.040229,35.75,7.69,28.06,13.305600,14.754400,9.0,22.50,36.0,22.305600,35.805600,49.305600,5.754400,-7.745600,-21.245600,True,False,False


missing per column: none
rows 9,935 → 12, columns 14 → 27   (code above reproduces the saved file exactly: True)


**How often did storing win?** [CALC from the gold table]

| | low (on-farm) | mid | high (commercial) |
|---|---|---|---|
| North Central | 6/6 | 0/6 | 0/6 |
| South Central | 6/6 | 1/6 | 0/6 |
| **Both regions** | **12/12** | **1/12** (2020 South Central) | **0/12** |
| Mean net gain (cents/bu) | +14.8 | −5.9 | −26.6 |

*Details:* [processing log: 2026-11-13: storage cost cases added to gold (06 → 07 → 08 rerun)](../docs/processing_log.md#2026-11-13-storage-cost-cases-added-to-gold-06--07--08-rerun) and [processing log: 2026-11-13: breakeven column, new bar chart, README storage item resolved](../docs/processing_log.md#2026-11-13-breakeven-column-new-bar-chart-readme-storage-item-resolved)

# Part 4: Linear regression and charts (`08_charts.py`)

**In plain words:** when cash is unusually far below futures in October (a big `gap_oct`, known on the decision date),
is more left over to pay for storage by April? "Left over" is `breakeven_storage_cents` = gap closed − interest: the most a
6-month storage could cost and storing still break even. If storage costs less than that, storing beat selling.

**a) Input: the gold table**, the columns the chart uses: x = `gap_oct_cents`, y = `breakeven_storage_cents`, and the
storage cost of each case.

In [22]:
show(gold[["decision_year", "region", "gap_oct_cents", "gap_closed_cents", "interest_cents",
           "breakeven_storage_cents", "storage_low_cents", "storage_mid_cents", "storage_high_cents"]])

12 rows x 9 columns


,decision_year,region,gap_oct_cents,gap_closed_cents,interest_cents,breakeven_storage_cents,storage_low_cents,storage_mid_cents,storage_high_cents
0,2020,North Central,29.80,21.91,0.224960,21.685040,9.0,22.25,35.5
1,2020,South Central,29.38,29.57,0.225209,29.344791,9.0,22.25,35.5
2,2021,North Central,40.56,20.00,0.153015,19.846985,9.0,22.50,36.0
3,2021,South Central,32.12,22.50,0.155608,22.344392,9.0,22.50,36.0
4,2022,North Central,35.75,28.06,13.305600,14.754400,9.0,22.50,36.0


missing per column: none


**b) Regression and check code.** `08_charts.py` **lines 77-80** (the fit), **lines 84-96** (pooled line and X) and
**lines 106-111** (dots above each storage case).

- `np.polyfit(x, y, 1)` fits the straight line `y = intercept + slope × x` by ordinary least squares, on all 12
  region-years together (pooled). R² is the squared correlation: the share of the ups and downs in y the line explains.
- **X** = the October gap where the line reaches the on-farm cost (9¢): `(9 − intercept) / slope`.
- **Check:** a dot above a storage case means storing beat selling, so the count must equal `store_wins` in gold (mid and
  high use each year's own cost). `08` stops if they differ.

In [23]:
def fit(x, y):                                                                 # 08 lines 77-80
    slope, intercept = np.polyfit(x, y, 1)                                     # OLS line: y = intercept + slope * x
    return slope, intercept, np.corrcoef(x, y)[0, 1] ** 2                      # R² = squared correlation
x_col, y_col = "gap_oct_cents", "breakeven_storage_cents"                      # line 84: x and y
slope, intercept, r2 = fit(gold[x_col], gold[y_col])                           # line 92: pooled, both regions
low_cost = raw_storage["storage_low_cents"].iloc[0]                            # line 94: on-farm cost, 9¢
x_at_low = (low_cost - intercept) / slope                                      # line 96: Oct gap where line = 9¢
checks = pd.DataFrame({"case": CASES,
    "dots above": [int((gold[y_col] > gold[f"storage_{c}_cents"]).sum()) for c in CASES],   # line 107
    "gold store_wins": [int(gold[f"store_wins_{c}"].sum()) for c in CASES]})                # line 108

**c) Results:** the pooled line, X, and the dots above each case next to the gold table's wins.

In [24]:
show(checks)
print(f"pooled OLS: breakeven = {intercept:.4f} + {slope:.4f} × gap_oct, R² = {r2:.4f}, n = {len(gold)}"
      f"   (processing log: 12.0949, 0.2517, 0.2950)")
print(f"X = {x_at_low:.1f}¢, observed Oct gaps {gold[x_col].min():.1f} to {gold[x_col].max():.1f}¢;"
      f"  dots above = gold wins: {checks['dots above'].eq(checks['gold store_wins']).all()}")

3 rows x 3 columns


,case,dots above,gold store_wins
0,low,12,12
1,mid,1,1
2,high,0,0


missing per column: none
pooled OLS: breakeven = 12.0949 + 0.2517 × gap_oct, R² = 0.2950, n = 12   (processing log: 12.0949, 0.2517, 0.2950)
X = -12.3¢, observed Oct gaps 29.4 to 66.6¢;  dots above = gold wins: True


**Reading it:**
- **Slope 0.25:** every extra 10¢ of October gap left about 2.5¢ more to pay for storage. But **R² is only 0.30**: the
  October gap explains less than a third of the variation, so it is a weak clue on its own.
- **Why weaker than gap closed alone** (slope 0.41, R² 0.67): interest is subtracted, and it jumped from about 0.2¢ (2020-2021)
  to 7-13¢ (2022-2025). The widest October gaps came in the high-rate years, so subtracting interest flattens the line.
- **No 9¢ threshold:** over the observed October gaps (29-67¢) the line runs from 19.5¢ to 28.9¢, always above the on-farm
  cost. It would reach 9¢ only at an October gap of −12.3¢, which never happens, so the chart marks no threshold: on-farm storage
  paid at every gap seen. The line enters the mid band only above an October gap of about 40¢ and never reaches the high band.
- It rests on **6 points per region**, so one unusual year can move it a lot (Part 5 shows exactly that).

### `outputs/scatter_gap.png`: the decision chart

Each blue dot is one Illinois region-year: October gap (x) against what was left to pay for storage (y). The gray line is the
pooled fit from the table above, drawn only over the observed gaps. The storage cases are the dotted line (low, on-farm, 9¢)
and two shaded bands (mid and high), each spanning its lowest to highest cost across the years. **A dot above a band beat
selling at that cost.** A dot inside a band depends on that year's own cost: only 2020 South Central beats the mid cost.
All 12 dots are above the low line; none reaches the high band.

![scatter_gap](../outputs/scatter_gap.png)

### `outputs/bars_by_year.png`

The green bar is the gap closed each year; the three short lines are the hurdles (storage + interest) for the low, mid and
high cases. Storing wins where the bar is **above** a line: always above the low (on-farm) line, almost never above the mid
and high lines. The hurdles jump in 2023, when both the commercial storage rate and the T-bill rate rose.

![bars_by_year](../outputs/bars_by_year.png)

*Details:* [processing log: 2026-11-21: 08 scatter replaced by a "decision" chart (08 rerun only)](../docs/processing_log.md#2026-11-21-08-scatter-replaced-by-a-decision-chart-08-rerun-only) and [processing log: 08_charts.py: charts](../docs/processing_log.md#08_chartspy-charts)

# Part 5: Walk-forward test (`09_walk_forward.py`)

Part 4 fitted a line on **all** years, which describes the data but cheats as a forecast: in October 2023 nobody knew the
2024 result. The walk-forward test predicts each test year using **only earlier years**, then compares with what really
happened. The two regions are pooled (2 rows per year).

| Round | Train on decision years | Training rows | Predict | Last training result known by | Decision date |
|---|---|---|---|---|---|
| 1 | 2020-2022 | 6 | 2023 | Apr 2023 | Oct 13, 2023 |
| 2 | 2020-2023 | 8 | 2024 | Apr 2024 | Oct 15, 2024 |
| 3 | 2020-2024 | 10 | 2025 | Apr 2025 | Oct 15, 2025 |

**Why there is no leakage:** a training year's `gap_closed` is known on Apr 15 of the next calendar year, before the test
year's Oct 15. The test row's `gap_oct`, interest and storage cost are all known on its Oct 15 decision date.

**a) Input: the gold table** (`09` reads `data/gold/gold_yearly.csv`).

In [25]:
show(gold)

12 rows x 27 columns


,decision_year,region,buyer_type,oct_date,oct_cash,oct_futures,apr_date,apr_cash,apr_futures,rate_decimal,gap_oct_cents,gap_apr_cents,gap_closed_cents,interest_cents,breakeven_storage_cents,storage_low_cents,storage_mid_cents,storage_high_cents,hurdle_low_cents,hurdle_mid_cents,hurdle_high_cents,net_gain_low_cents,net_gain_mid_cents,net_gain_high_cents,store_wins_low,store_wins_mid,store_wins_high
0,2020,North Central,Country Elevators,2020-10-15,3.7945,4.0925,2021-04-15,5.8211,5.9000,0.001186,29.80,7.89,21.91,0.224960,21.685040,9.0,22.25,35.5,9.224960,22.474960,35.724960,12.685040,-0.564960,-13.814960,True,False,False
1,2020,South Central,Country Elevators,2020-10-15,3.7987,4.0925,2021-04-15,5.9019,5.9000,0.001186,29.38,-0.19,29.57,0.225209,29.344791,9.0,22.25,35.5,9.225209,22.475209,35.725209,20.344791,7.094791,-6.155209,True,True,False
2,2021,North Central,Country Elevators,2021-10-15,4.9819,5.3875,2022-04-14,7.6969,7.9025,0.000614,40.56,20.56,20.00,0.153015,19.846985,9.0,22.50,36.0,9.153016,22.653015,36.153016,10.846985,-2.653015,-16.153015,True,False,False
3,2021,South Central,Country Elevators,2021-10-15,5.0663,5.3875,2022-04-14,7.8063,7.9025,0.000614,32.12,9.62,22.50,0.155608,22.344392,9.0,22.50,36.0,9.155608,22.655608,36.155608,13.344392,-0.155608,-13.655608,True,False,False
4,2022,North Central,Country Elevators,2022-10-14,6.6150,6.9725,2023-04-14,6.5856,6.6625,0.040229,35.75,7.69,28.06,13.305600,14.754400,9.0,22.50,36.0,22.305600,35.805600,49.305600,5.754400,-7.745600,-21.245600,True,False,False


missing per column: none


**b) Forecast, decision and scoring code.** `09_walk_forward.py` **lines 39-65** (forecasts), **lines 67-73** (decisions)
and **lines 75-90** (scoreboard); shortened to fewer stored columns.

- **Two forecasts of `gap_closed`** for each test row: **regression** (fit `gap_closed ~ gap_oct` on the training rows,
  apply it to the test row's `gap_oct`) and **past average** (mean `gap_closed` of the training rows).
- **Rule** for storage case c: **store if forecast − interest > storage_c**, otherwise sell. Every input is known on Oct 15.
- **Score:** a stored row earns its real `net_gain_c`; a sold row earns 0. A **correct call** matches hindsight (store if the
  real net gain was > 0). MAE = average size of the forecast miss, in cents/bu.

In [26]:
rows = []                                                                      # one row per test row
for rnd, test_year in enumerate([2023, 2024, 2025], start=1):                  # lines 41-43: 3 rounds
    train = gold[gold["decision_year"] < test_year]                            # line 42: ONLY earlier years
    test = gold[gold["decision_year"] == test_year]                            # line 43: the year to predict
    slope, intercept = np.polyfit(train["gap_oct_cents"], train["gap_closed_cents"], 1)   # line 45: OLS on training rows
    avg = train["gap_closed_cents"].mean()                                     # line 46: past-average forecast
    print(f"round {rnd}: train {len(train)} rows → {test_year}: intercept {intercept:.2f}, slope {slope:.3f}, past average {avg:.2f}")
    for _, r in test.iterrows():                                               # lines 50-62: each test row
        rows.append({"round": rnd, "test_year": test_year, "region": r["region"],
                     "actual": r["gap_closed_cents"],                          # what really closed
                     "pred_regression": intercept + slope * r["gap_oct_cents"],   # line applied to this Oct gap
                     "pred_past_average": avg, "interest": r["interest_cents"],
                     **{f"storage_{c}": r[f"storage_{c}_cents"] for c in CASES},
                     **{f"net_gain_{c}": r[f"net_gain_{c}_cents"] for c in CASES}})
pred = pd.DataFrame(rows)
mae = {m: (pred[f"pred_{m}"] - pred["actual"]).abs().mean() for m in ["regression", "past_average"]}   # lines 64-65, 100-101
print(f"MAE over {len(pred)} test rows: regression {mae['regression']:.2f}, past average {mae['past_average']:.2f}  (log: 7.16 vs 8.38)")
score = []                                                                     # lines 76-90: scoreboard rows
for c in CASES:                                                                # each storage case
    best = pred[f"net_gain_{c}"] > 0                                           # line 70: hindsight best = store?
    calls = {"always sell": pd.Series(False, index=pred.index),                 # never store
             "always store": pd.Series(True, index=pred.index),                # store every year
             "past average": pred["pred_past_average"] - pred["interest"] > pred[f"storage_{c}"],   # line 72
             "regression": pred["pred_regression"] - pred["interest"] > pred[f"storage_{c}"]}       # line 73
    for s, store in calls.items():                                             # each strategy
        net = np.where(store, pred[f"net_gain_{c}"], 0.0)                      # line 83: real gain if stored, else 0
        score.append({"storage_case": c, "strategy": s, "times_stored": int(store.sum()),
                      "total_net_gain_cents": round(net.sum(), 2),             # line 86
                      "correct_calls": int((store == best).sum())})            # line 89
score = pd.DataFrame(score)

round 1: train 6 rows → 2023: intercept 34.38, slope -0.264, past average 25.45
round 2: train 8 rows → 2024: intercept 18.49, slope 0.215, past average 26.91
round 3: train 10 rows → 2025: intercept 11.25, slope 0.418, past average 29.91
MAE over 6 test rows: regression 7.16, past average 8.38  (log: 7.16 vs 8.38)


**c) Scoreboard:** total net gain over the 6 test rows (cents/bu) and correct calls out of 6, per storage case and strategy.

In [27]:
show(score.pivot(index="storage_case", columns="strategy", values="total_net_gain_cents").loc[list(CASES)].astype(str)
     + " (" + score.pivot(index="storage_case", columns="strategy", values="correct_calls").loc[list(CASES)].astype(str) + "/6)")
print("matches outputs/scoreboard.csv:", np.allclose(score["total_net_gain_cents"], scoreboard["total_net_gain_cents"], atol=0.01)
      and score["correct_calls"].eq(scoreboard["correct_calls"]).all())

3 rows x 4 columns


strategy,always sell,always store,past average,regression
storage_case,,,,
low,0.0 (0/6),106.5 (6/6),106.5 (6/6),86.53 (4/6)
mid,0.0 (6/6),-61.5 (0/6),0.0 (6/6),0.0 (6/6)
high,0.0 (6/6),-229.5 (0/6),0.0 (6/6),0.0 (6/6)


missing per column: none
matches outputs/scoreboard.csv: True


**Why round 1 got 2023 wrong.** The first line was fitted on only **6 points** (2020-2022), and its slope came out
**negative** (−0.26): a wider October gap predicted *less* gap closed, the opposite of the full-sample pattern (all 12 points: slope +0.41;
Part 4's chart shows the same upward tilt).
2023 had a wide October gap (54-57¢), so the line forecast only 19.3¢ (North Central) and 20.2¢ (South Central), just below
the low-case hurdle of interest + on-farm storage (21.3¢). The rule said **sell**, but the gap really closed 33.0¢ and 29.6¢:
storing would have earned +11.8¢ and +8.2¢, about 20¢ in total. The past average (25.5¢) was above the hurdle and said store.

**Results in short:**
- **Accuracy:** the regression missed by less on average (MAE **7.16** vs **8.38** cents/bu).
- **Money:** in the **low** case, *always store* and *past average* both earned **+106.5¢** over the 6 test rows (6/6
  correct), while the regression earned **+86.5¢** (4/6), because of the 2023 mistake. In the **mid** and **high** cases
  storing always lost (−61.5¢ and −229.5¢ for always store), and every forecast rule correctly said sell.
- **Lesson:** a more accurate forecast does not guarantee better decisions. Only errors near the hurdle change the call,
  and the regression's one big error happened right there.

*Details:* [processing log: 2026-11-13: 09 walk-forward test (first out-of-sample results)](../docs/processing_log.md#2026-11-13-09-walk-forward-test-first-out-of-sample-results)

# Part 6: Wrap-up

## What this means

- **Storage cost decides the answer, more than forecasting does.** With on-farm storage (about 9¢ for 6 months), a hedged
  storage beat selling at harvest in all 12 region-years. At commercial elevator rates it never did, and at the mid
  cost only once.
- **The October gap did close every year.** The gap between May futures and the local cash price narrowed by 20-43¢
  from October to April, always more than the T-bill interest given up (0.2-13¢). The question is only whether it also covers storage.
- **A simple rule was as good as the model.** "Store if the past average beats the hurdle" made every correct call in the
  test years; the regression had a smaller forecast error but made a costly wrong call in 2023, because its first
  line rested on 6 points.

## Limits and next steps

- **Small sample:** 6 decision years × 2 regions, only 3 test years (6 test rows). Results could easily be luck, and
  round 1's regression was fitted on 6 points. Next: **more regions**: clean bids for 6 states are already in the pipeline
  (section 1.1); the gold table and the walk-forward test still use Illinois only. **More years** (back to 2010) need
  another source, because the USDA API starts in mid-2020.
- **Storage-cost assumptions:** the commercial (high) cost is one Ohio elevator's yearly rate halved, which probably
  understates a 6-month hold (about 68% of the yearly cost, not 50%); the on-farm (low) cost is a single 2016 range,
  the same every year; 2025 high is carried forward from 2024. Shrink, commissions and margin interest are ignored.
- **Apr-vs-Oct basis check:** compare the April basis with the October basis directly, not only their difference, to
  see whether storing gains come from a weak October or a strong April.